In [9]:
import pandas as pd
import json
import re
from pathlib import Path

# Excel 文件匹配模式：period_1.xlsx, period_2.xlsx ...
PATTERN = "period_*.xlsx"


def clean_result_text(txt):
    """
    清洗 result 字段：
    1. 去掉所有 "--"
    2. 去掉前后空格
    3. 最后补 "元"
    例：
      一等奖--800   → 一等奖800元
      三等奖--100   → 三等奖100元
    """
    if txt is None:
        return ""

    # 去掉 --
    cleaned = str(txt).replace("--", "").strip()

    # 如果末尾没有 '元'，加上
    if not cleaned.endswith("元"):
        cleaned = cleaned + "元"

    return cleaned


def excel_to_json_multi():
    root = Path(".")
    excel_files = sorted(root.glob(PATTERN))

    if not excel_files:
        print(f"当前目录未找到匹配 {PATTERN} 的 Excel 文件")
        return

    periods_meta = []

    for excel_path in excel_files:
        name = excel_path.name
        print(f"\n处理文件：{name}")

        # 从文件名获取期数
        m = re.match(r"period_(.+)\.xlsx$", name)
        if not m:
            print(f"  跳过：文件名无法解析期数：{name}")
            continue

        period_id = m.group(1)

        # 读取 Excel
        df = pd.read_excel(excel_path, dtype=str)

        # 校验字段
        if "fid" not in df.columns or "result" not in df.columns:
            print(f"  错误：Excel 中必须包含 fid、result 两列")
            print("  当前列名：", df.columns.tolist())
            continue

        data = {}

        for _, row in df.iterrows():
            fid = str(row["fid"]).strip()
            raw_result = row["result"]

            if not fid:
                continue

            # 清洗 result
            cleaned_result = clean_result_text(raw_result)

            data[fid] = cleaned_result

        # 输出 json 文件
        json_filename = f"data_period_{period_id}.json"
        json_path = root / json_filename

        with json_path.open("w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        print(f"  已生成：{json_filename}（{len(data)} 条记录）")

        periods_meta.append({"id": str(period_id), "label": f"第{period_id}期"})

    # 排序期数
    def sort_key(p):
        try:
            return int(p["id"])
        except:
            return p["id"]

    periods_meta.sort(key=sort_key)

    # 写入 periods.json
    periods_json_path = root / "periods.json"
    with periods_json_path.open("w", encoding="utf-8") as f:
        json.dump(periods_meta, f, ensure_ascii=False, indent=2)

    print(f"\n所有期数处理完毕，已生成 periods.json：{periods_json_path.resolve()}")
    print("期数列表：")
    for p in periods_meta:
        print(f"  id={p['id']}, label={p['label']}")


if __name__ == "__main__":
    excel_to_json_multi()


处理文件：period_1.xlsx
  已生成：data_period_1.json（44622 条记录）

处理文件：period_2.xlsx
  已生成：data_period_2.json（44622 条记录）

所有期数处理完毕，已生成 periods.json：/Users/rr/jiangsucha-web/periods.json
期数列表：
  id=1, label=第1期
  id=2, label=第2期
